# AD / MCI / HC 3그룹 — EEGNet 이진분류 (김동빈 연구원 요청)

**요청 원문** (카카오톡, 2026-09 오후):
> [김동빈] ad mci hc 파일인데, eegnet으로 정확도, 민감도, 특이도, f1-score 포스터 만드는동안 뽑아보세요.
> [강태우] 그러면 지금 vmci-hc 관련한 부분만 민감도, 특이도, f1-score만 뽑으면 되는걸까요? 초록에는 이 부분만 들어가서요
> [김동빈] 네네
> [강태우] 데이터수들이 더 많은데 그럼 이걸로 지금 학습 과정을 다시 진행해보라는 말씀이실까요? 원래 주신 데이터셋들 말고요
> [김동빈] 이번에 한 주제랑 다른거고, vd보다는 메이저한 레이블로 구성된 데이터로 eegnet 한 번 돌려보라는 의미였습니다.

**해석**: 현재 진행 중인 VMCI-HC 포스터(혈관성 소그룹)와는 별개로, CAUEEG의 표준/메이저 진단 레이블인
**AD, MCI, HC** 전체 그룹으로 EEGNet을 돌려 정확도·민감도·특이도·F1-score를 뽑는 사이드 태스크.
이진분류 지표(민감도/특이도)를 요청했으므로 **AD vs HC**, **MCI vs HC** 두 개의 이진분류로 진행.

**데이터 검증 완료** — `CAUEEG_annotation.xlsx`의 `annotation` 시트와 실제 폴더 구성을 subject 단위로 전수
대조한 결과:
- `ad/` 폴더(212명) = annotation `ad==1` (100% 일치, `dementia==1`이기도 함)
- `mci/` 폴더(305명) = annotation `mci==1` (100% 일치)
- `HC/` 폴더(246명) = annotation `normal==1` (100% 일치)
- 세 그룹 간 serial 중복 없음(disjoint) — 라벨 오염 없음

**이번 노트북의 범위**: 요청받은 4개 지표(Accuracy/Sensitivity/Specificity/F1)만 산출. 기존 vascular
연구처럼 채널/대역 ablation, qEEG 비교 등은 포함하지 않음(필요해지면 별도로 추가 예정) — vascular
파이프라인(`01_Vascular_mci_hc_vd.ipynb`)에서 이미 검증된 전처리/모델/학습 코드를 그대로 재사용.

⚠️ **CV 설정 가정**: 요청에 fold/repeat 수가 명시되지 않아, 우선 **5-fold × 1-repeat × 100epoch**으로
설정함(vascular 연구보다 표본이 3~4배 많아 5-repeat까지 돌리면 훨씬 오래 걸릴 것으로 예상 — 정확한 소요
시간은 1개 fold 완료 후 실측치로 다시 안내 예정). repeat을 늘리고 싶으면 `N_REPEATS`만 바꾸면 됨.

## 1. 경로 설정 / 상수 / 임포트

In [ ]:
import os, glob, json
from datetime import datetime
import numpy as np, pandas as pd, mne
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, recall_score

# 압축 해제 위치 (연구실 컴퓨터 기준) — 다른 컴퓨터에서 돌릴 때는 이 경로만 수정
DATA_ROOT_ADMCIHC = "C:/eeg_research/CAUEEG_all/CAUEEG/"
GROUP_DIRS_ADMCIHC = {'ad': 'ad', 'mci': 'mci', 'hc': 'HC'}
RESULTS_DIR = DATA_ROOT_ADMCIHC + 'results_ad_mci_hc/'
os.makedirs(RESULTS_DIR, exist_ok=True)

FS_TARGET = 250
WIN_SEC = 4
WIN_STRIDE_SEC = 2
VAL_FRACTION = 0.15
BATCH_SIZE = 32
LR = 1e-3
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE =', DEVICE)

N_FOLDS = 5
N_REPEATS = 1     # 가정값 — 필요시 조정 (마크다운 설명 참고)
EPOCHS = 100

# select_channels/build_windows_for_experiment가 참조하는 최소 스텁(이번 노트북은 전체 채널만 사용)
CHANNEL_REGIONS = {'all': None}
BANDS = {'broadband': None}

## 2. 전처리 유틸 함수

`01_Vascular_mci_hc_vd.ipynb` Cell 4-2와 동일(검증된 코드 그대로 재사용).

In [ ]:
from scipy.signal import resample_poly, butter, filtfilt

def bandpass(x, low, high, fs, order=4):
    b, a = butter(order, [low, high], btype='bandpass', fs=fs)
    return filtfilt(b, a, x, axis=-1)

def minimal_preprocess(x, fs_orig, fs_target=None, band=None):
    if fs_target is None:
        fs_target = FS_TARGET
    if fs_orig != fs_target:
        x = resample_poly(x, fs_target, fs_orig, axis=-1)
    if band is not None:
        low, high = band
        x = bandpass(x, low, high, fs_target)
    x = (x - x.mean(axis=-1, keepdims=True)) / (x.std(axis=-1, keepdims=True) + 1e-12)
    return x.astype(np.float32)

def make_windows(x, fs, win_sec=None, stride_sec=None):
    if win_sec is None:
        win_sec = WIN_SEC
    if stride_sec is None:
        stride_sec = WIN_STRIDE_SEC
    win_len, stride = int(win_sec * fs), int(stride_sec * fs)
    n = x.shape[-1]
    out, start = [], 0
    while start + win_len <= n:
        out.append(x[:, start:start + win_len])
        start += stride
    return out

def select_channels(x, ch_names, region):
    if region is None:
        return x, ch_names
    keep_names = CHANNEL_REGIONS[region]
    if keep_names is None:
        return x, ch_names
    idx = [i for i, name in enumerate(ch_names) if name in keep_names]
    if len(idx) == 0:
        raise RuntimeError(f'region={region}: ch_names={ch_names} 안에 매칭되는 채널이 없음')
    return x[idx, :], [ch_names[i] for i in idx]

## 3. Dataset 클래스 + EEGNet 모델

`01_Vascular_mci_hc_vd.ipynb` Cell 4-3과 동일(검증된 코드 그대로 재사용).

In [ ]:
class EEGWindowDataset(Dataset):
    def __init__(self, all_windows, all_labels, indices):
        self.all_windows = all_windows
        self.all_labels = all_labels
        self.indices = indices
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        idx = self.indices[i]
        x = torch.tensor(self.all_windows[idx], dtype=torch.float32).unsqueeze(0)
        y = torch.tensor(self.all_labels[idx], dtype=torch.long)
        return x, y

class EEGNet(nn.Module):
    def __init__(self, n_channels, n_samples, n_classes, F1=8, D=2, F2=16, kernel_len=64, dropout=0.5):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, kernel_len), padding=(0, kernel_len // 2), bias=False),
            nn.BatchNorm2d(F1),
            nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D), nn.ELU(), nn.AvgPool2d((1, 4)), nn.Dropout(dropout),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(F1 * D, F1 * D, (1, 16), padding=(0, 8), groups=F1 * D, bias=False),
            nn.Conv2d(F1 * D, F2, (1, 1), bias=False),
            nn.BatchNorm2d(F2), nn.ELU(), nn.AvgPool2d((1, 8)), nn.Dropout(dropout),
        )
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_samples)
            flat_dim = self.block2(self.block1(dummy)).numel()
        self.classifier = nn.Linear(flat_dim, n_classes)
    def forward(self, x):
        x = self.block2(self.block1(x))
        return self.classifier(x.flatten(1))

## 4. 학습 루프 + CV 실행

`01_Vascular_mci_hc_vd.ipynb` Cell 4-4(Sensitivity/Specificity 포함 최신 버전)와 동일(검증된 코드
그대로 재사용). `tag`만 이번 노트북에서 `'ad_vs_hc'` / `'mci_vs_hc'`로 다르게 넣어서 호출.

In [ ]:
def train_one_fold(model, train_loader, val_loader, epochs=None, lr=None, class_weights=None):
    if epochs is None:
        epochs = EPOCHS
    if lr is None:
        lr = LR
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE) if class_weights is not None else None)
    best_state, best_val_loss = None, float('inf')
    for _ in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            crit(model(x), y).backward()
            opt.step()
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                val_loss += crit(model(x), y).item() * x.size(0)
        val_loss /= len(val_loader.dataset)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model

@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    probs, ys = [], []
    for x, y in loader:
        p = F.softmax(model(x.to(DEVICE)), dim=1).cpu().numpy()
        probs.append(p)
        ys.append(y.numpy())
    return np.concatenate(probs), np.concatenate(ys)

def run_cv(all_windows, all_labels, all_subjects, n_channels, n_samples, n_classes,
           n_folds=None, n_repeats=None, epochs=None, save_path='cv_results.json', tag=''):
    if n_folds is None:
        n_folds = N_FOLDS
    if n_repeats is None:
        n_repeats = N_REPEATS
    if epochs is None:
        epochs = EPOCHS
    all_labels_arr = np.array(all_labels)
    all_subjects_arr = np.array(all_subjects)
    window_aucs, subj_accs, subj_f1s, subj_sens, subj_spec = [], [], [], [], []
    fold_records = []

    for rep in range(n_repeats):
        sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=rep)
        for fold, (trainval_idx, te_idx) in enumerate(
                sgkf.split(all_windows, all_labels_arr, groups=all_subjects_arr)):
            trainval_subjects_arr = all_subjects_arr[trainval_idx]
            gss = GroupShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=1000 + rep * 100 + fold)
            train_rel, val_rel = next(gss.split(trainval_idx, groups=trainval_subjects_arr))
            train_idx = trainval_idx[train_rel]
            val_idx = trainval_idx[val_rel]

            train_loader = DataLoader(EEGWindowDataset(all_windows, all_labels, train_idx), batch_size=BATCH_SIZE, shuffle=True)
            val_loader = DataLoader(EEGWindowDataset(all_windows, all_labels, val_idx), batch_size=BATCH_SIZE, shuffle=False)
            test_loader = DataLoader(EEGWindowDataset(all_windows, all_labels, te_idx), batch_size=BATCH_SIZE, shuffle=False)

            train_labels_arr = all_labels_arr[train_idx]
            class_counts = np.bincount(train_labels_arr, minlength=n_classes).astype(np.float64)
            class_counts[class_counts == 0] = 1.0
            class_weights = torch.tensor(class_counts.sum() / (n_classes * class_counts), dtype=torch.float32)

            model = EEGNet(n_channels, n_samples, n_classes).to(DEVICE)
            model = train_one_fold(model, train_loader, val_loader, epochs=epochs, class_weights=class_weights)
            probs, ys = predict_proba(model, test_loader)

            fold_auc = roc_auc_score(ys, probs[:, 1]) if n_classes == 2 else roc_auc_score(ys, probs, multi_class='ovr')
            window_aucs.append(fold_auc)

            test_subjects = all_subjects_arr[te_idx]
            subj_true, subj_pred = [], []
            for subj in np.unique(test_subjects):
                mask = test_subjects == subj
                subj_pred.append(probs[mask].mean(axis=0).argmax())
                subj_true.append(ys[mask][0])
            subj_accs.append(accuracy_score(subj_true, subj_pred))
            subj_f1s.append(f1_score(subj_true, subj_pred, average='macro'))
            subj_sens.append(recall_score(subj_true, subj_pred, pos_label=1, zero_division=0))
            subj_spec.append(recall_score(subj_true, subj_pred, pos_label=0, zero_division=0))

            fold_records.append({
                'repeat': rep, 'fold': fold, 'window_auc': float(fold_auc),
                'subj_acc': float(subj_accs[-1]), 'subj_macro_f1': float(subj_f1s[-1]),
                'subj_sensitivity': float(subj_sens[-1]), 'subj_specificity': float(subj_spec[-1]),
                'n_train_windows': int(len(train_idx)), 'n_val_windows': int(len(val_idx)),
                'n_test_windows': int(len(te_idx)), 'n_test_subjects': int(len(np.unique(test_subjects))),
            })

            with open(save_path, 'w', encoding='utf-8') as f:
                json.dump({
                    'tag': tag,
                    'config': {'n_folds': n_folds, 'n_repeats': n_repeats, 'epochs': epochs,
                               'win_sec': WIN_SEC, 'win_stride_sec': WIN_STRIDE_SEC, 'fs_target': FS_TARGET,
                               'lr': LR, 'n_channels': n_channels, 'n_samples': n_samples},
                    'folds': fold_records,
                    'summary_so_far': {
                        'window_auc_mean': float(np.mean(window_aucs)), 'window_auc_std': float(np.std(window_aucs)),
                        'subj_acc_mean': float(np.mean(subj_accs)), 'subj_acc_std': float(np.std(subj_accs)),
                        'subj_macro_f1_mean': float(np.mean(subj_f1s)), 'subj_macro_f1_std': float(np.std(subj_f1s)),
                        'subj_sensitivity_mean': float(np.mean(subj_sens)), 'subj_sensitivity_std': float(np.std(subj_sens)),
                        'subj_specificity_mean': float(np.mean(subj_spec)), 'subj_specificity_std': float(np.std(subj_spec)),
                    },
                    'saved_at': datetime.now().isoformat(),
                }, f, indent=2, ensure_ascii=False)

            print(f'[{tag}][rep {rep+1}/{n_repeats}][fold {fold+1}/{n_folds}] '
                  f'AUC={fold_auc:.3f} subj_acc={subj_accs[-1]:.3f} subj_f1={subj_f1s[-1]:.3f} '
                  f'sens={subj_sens[-1]:.3f} spec={subj_spec[-1]:.3f} '
                  f'(train={len(train_idx)} val={len(val_idx)} test={len(te_idx)} windows, '
                  f'test_subj={len(np.unique(test_subjects))}) -> {save_path}')

    summary = {
        'window_auc_mean': float(np.mean(window_aucs)), 'window_auc_std': float(np.std(window_aucs)),
        'subj_acc_mean': float(np.mean(subj_accs)), 'subj_acc_std': float(np.std(subj_accs)),
        'subj_macro_f1_mean': float(np.mean(subj_f1s)), 'subj_macro_f1_std': float(np.std(subj_f1s)),
        'subj_sensitivity_mean': float(np.mean(subj_sens)), 'subj_sensitivity_std': float(np.std(subj_sens)),
        'subj_specificity_mean': float(np.mean(subj_spec)), 'subj_specificity_std': float(np.std(subj_spec)),
    }
    print(f"\n[{tag}] window AUC={summary['window_auc_mean']:.3f}±{summary['window_auc_std']:.3f}  "
          f"subj_acc={summary['subj_acc_mean']:.3f}±{summary['subj_acc_std']:.3f}  "
          f"subj_macroF1={summary['subj_macro_f1_mean']:.3f}±{summary['subj_macro_f1_std']:.3f}  "
          f"sens={summary['subj_sensitivity_mean']:.3f}±{summary['subj_sensitivity_std']:.3f}  "
          f"spec={summary['subj_specificity_mean']:.3f}±{summary['subj_specificity_std']:.3f}")
    return summary

## 5. 데이터 로딩 (ad / mci / HC 폴더 직접 스캔)

`subject_list.csv` 없이 `ad/`, `mci/`, `HC/` 폴더를 직접 스캔 — 폴더 소속 자체가 라벨이므로
annotation 파일과의 대조는 이미 위(1절)에서 전수 검증 완료. subject별 `.set` 파일 1개씩 로딩
(`p1_CAUEEG_{serial}_merged.set` 형식, `.fdt` 없이 자체완결형).

⚠️ 이 셀이 이번 노트북에서 가장 오래 걸리는 셀(763명 raw EEG 로딩) — 한 번 실행 후 `cache_admcihc`는
메모리에 유지되므로, 아래 실행 셀들을 재실행할 때 이 셀은 다시 돌릴 필요 없음(커널을 껐다 켰을 때만 재실행).

In [ ]:
def load_all_subjects_admcihc():
    mne.set_log_level('ERROR')
    cache = {}
    ch_names = None
    failed = []
    for group, dirname in GROUP_DIRS_ADMCIHC.items():
        group_root = os.path.join(DATA_ROOT_ADMCIHC, dirname)
        subj_folders = sorted(os.listdir(group_root))
        for folder in subj_folders:
            subj_dir = os.path.join(group_root, folder)
            if not os.path.isdir(subj_dir):
                continue
            matches = glob.glob(os.path.join(subj_dir, '*.set'))
            if not matches:
                failed.append((group, folder))
                continue
            raw = mne.io.read_raw_eeglab(matches[0], preload=True)
            raw.pick_types(eeg=True)
            if ch_names is None:
                ch_names = raw.ch_names
                print(f'첫 subject 기준: 채널 {len(ch_names)}개 {ch_names}, sfreq={raw.info["sfreq"]}Hz')
            elif set(raw.ch_names) != set(ch_names):
                print(f'!! 경고: {folder}({group}) 채널 구성 자체가 기준과 다름 -> 스킵')
                failed.append((group, folder))
                continue
            elif raw.ch_names != ch_names:
                raw.reorder_channels(ch_names)
            sfreq = raw.info['sfreq']
            x = raw.get_data()
            del raw
            x = resample_poly(x, FS_TARGET, sfreq, axis=-1).astype(np.float32) if sfreq != FS_TARGET else x.astype(np.float32)
            cache[f'{group}_{folder}'] = {'x': x, 'group': group}
            del x
    counts = {}
    for v in cache.values():
        counts[v['group']] = counts.get(v['group'], 0) + 1
    print(f'로딩 완료: 총 {len(cache)}명 {counts}, 실패={len(failed)}건 {failed}')
    return cache, ch_names

cache_admcihc, ch_names_admcihc = load_all_subjects_admcihc()

## 6. 그룹쌍 선택 + 윈도우 생성 함수

vascular 파이프라인의 `select_group_pair` / `build_windows_for_experiment`와 동일(재사용).
`group_pos`가 label=1, `group_neg`가 label=0.

In [ ]:
def select_group_pair(cache, group_pos, group_neg):
    sub_cache = {}
    for sid, d in cache.items():
        if d['group'] == group_pos:
            sub_cache[sid] = {'x': d['x'], 'label': 1}
        elif d['group'] == group_neg:
            sub_cache[sid] = {'x': d['x'], 'label': 0}
    n_pos = sum(v['label'] == 1 for v in sub_cache.values())
    n_neg = sum(v['label'] == 0 for v in sub_cache.values())
    print(f'그룹쌍 선택: {group_pos}(label=1, n={n_pos}) vs {group_neg}(label=0, n={n_neg}), 총 {len(sub_cache)}명')
    return sub_cache

def build_windows_for_experiment(cache, ch_names, band=None, region=None):
    all_windows, all_labels, all_subjects = [], [], []
    band_range = BANDS[band] if band else None
    for sub_id, d in cache.items():
        x_sel, _ = select_channels(d['x'], ch_names, region)
        x_proc = minimal_preprocess(x_sel, fs_orig=FS_TARGET, fs_target=FS_TARGET, band=band_range)
        for w in make_windows(x_proc, fs=FS_TARGET):
            all_windows.append(w)
            all_labels.append(d['label'])
            all_subjects.append(sub_id)
    return all_windows, all_labels, all_subjects

## 7. 실행: AD vs HC (EEGNet, 19채널 broadband, 5-fold × 1-repeat × 100epoch)

In [ ]:
pair_ad_hc = select_group_pair(cache_admcihc, 'ad', 'hc')
windows_ad_hc, labels_ad_hc, subjects_ad_hc = build_windows_for_experiment(pair_ad_hc, ch_names_admcihc)
n_channels_ad_hc = windows_ad_hc[0].shape[0]
n_samples_ad_hc = windows_ad_hc[0].shape[1]
print(f'AD vs HC: {len(windows_ad_hc)}개 윈도우, {n_channels_ad_hc}채널 x {n_samples_ad_hc}샘플')

summary_ad_hc = run_cv(
    windows_ad_hc, labels_ad_hc, subjects_ad_hc,
    n_channels=n_channels_ad_hc, n_samples=n_samples_ad_hc, n_classes=2,
    n_folds=N_FOLDS, n_repeats=N_REPEATS, epochs=EPOCHS,
    save_path=RESULTS_DIR + 'ad_vs_hc_eegnet.json', tag='ad_vs_hc',
)

## 8. 실행: MCI vs HC (EEGNet, 19채널 broadband, 5-fold × 1-repeat × 100epoch)

In [ ]:
pair_mci_hc = select_group_pair(cache_admcihc, 'mci', 'hc')
windows_mci_hc, labels_mci_hc, subjects_mci_hc = build_windows_for_experiment(pair_mci_hc, ch_names_admcihc)
n_channels_mci_hc = windows_mci_hc[0].shape[0]
n_samples_mci_hc = windows_mci_hc[0].shape[1]
print(f'MCI vs HC: {len(windows_mci_hc)}개 윈도우, {n_channels_mci_hc}채널 x {n_samples_mci_hc}샘플')

summary_mci_hc = run_cv(
    windows_mci_hc, labels_mci_hc, subjects_mci_hc,
    n_channels=n_channels_mci_hc, n_samples=n_samples_mci_hc, n_classes=2,
    n_folds=N_FOLDS, n_repeats=N_REPEATS, epochs=EPOCHS,
    save_path=RESULTS_DIR + 'mci_vs_hc_eegnet.json', tag='mci_vs_hc',
)

## 9. 결과 요약 표 (Accuracy / Sensitivity / Specificity / F1-score)

김동빈 연구원이 요청한 4개 지표만 정리. `results_ad_mci_hc/*.json`의 `summary_so_far`(=최종 summary와
동일, 매 fold마다 누적 갱신됨)를 그대로 읽어서 표로 구성 — 재실행 없이 이 셀만 다시 돌려도 됨.

In [ ]:
def load_summary(tag):
    path = RESULTS_DIR + f'{tag}_eegnet.json'
    with open(path, encoding='utf-8') as f:
        d = json.load(f)
    return d['summary_so_far'], d['config']

rows = []
for tag, label in [('ad_vs_hc', 'AD vs HC'), ('mci_vs_hc', 'MCI vs HC')]:
    s, cfg = load_summary(tag)
    rows.append({
        '비교': label,
        'Accuracy(%)': f"{s['subj_acc_mean']*100:.1f} ± {s['subj_acc_std']*100:.1f}",
        'Sensitivity(%)': f"{s['subj_sensitivity_mean']*100:.1f} ± {s['subj_sensitivity_std']*100:.1f}",
        'Specificity(%)': f"{s['subj_specificity_mean']*100:.1f} ± {s['subj_specificity_std']*100:.1f}",
        'F1-score(%, macro)': f"{s['subj_macro_f1_mean']*100:.1f} ± {s['subj_macro_f1_std']*100:.1f}",
        'AUC': f"{s['window_auc_mean']:.3f} ± {s['window_auc_std']:.3f}",
        'n_folds x n_repeats': f"{cfg['n_folds']} x {cfg['n_repeats']}",
    })

df_summary = pd.DataFrame(rows)
df_summary.to_csv(RESULTS_DIR + 'ad_mci_hc_summary.csv', index=False, encoding='utf-8-sig')
df_summary